In [0]:
# Run Setup notebook to load raw df and full_path variable
VOLUME_CSV_PATH = "/Volumes/students_data/team4-chicago/chicago-restaurants"
file_name = "Food_Inspections.csv"
full_path = f"{VOLUME_CSV_PATH}/{file_name}"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType

# Explicit schema with enforced data types and nullability
# nullable=False columns will be enforced as NOT NULL when written to Delta
inspection_schema = StructType([
    StructField("Inspection ID",   IntegerType(), nullable=False),
    StructField("DBA Name",        StringType(),  nullable=False),
    StructField("AKA Name",        StringType(),  nullable=True),   # 2,418 legitimate nulls
    StructField("License #",       IntegerType(), nullable=False),
    StructField("Facility Type",   StringType(),  nullable=False),
    StructField("Risk",            StringType(),  nullable=False),
    StructField("Address",         StringType(),  nullable=False),
    StructField("City",            StringType(),  nullable=False),
    StructField("State",           StringType(),  nullable=False),
    StructField("Zip",             IntegerType(), nullable=False),
    StructField("Inspection Date", DateType(),     nullable=False),
    StructField("Inspection Type", StringType(),  nullable=False),
    StructField("Results",         StringType(),  nullable=False),
    StructField("Violations",      StringType(),  nullable=True),   # 28% null — no violations found
    StructField("Latitude",        DoubleType(),  nullable=True),   # not always geocoded
    StructField("Longitude",       DoubleType(),  nullable=True),   # not always geocoded
    StructField("Location",        StringType(),  nullable=True),   # not always geocoded
])

In [0]:
# Read CSV with explicit schema (types enforced, nullable ignored by CSV reader)
df = spark.read.csv(
    full_path,
    header=True,
    schema=inspection_schema,
    multiLine=True,
    escape='"',
    dateFormat="M/d/yyyy"
)

# Rename columns to Delta-compatible snake_case (no spaces or special chars)
df_clean = df \
    .withColumnRenamed("Inspection ID", "inspection_id") \
    .withColumnRenamed("DBA Name", "dba_name") \
    .withColumnRenamed("AKA Name", "aka_name") \
    .withColumnRenamed("License #", "license_num") \
    .withColumnRenamed("Facility Type", "facility_type") \
    .withColumnRenamed("Risk", "risk") \
    .withColumnRenamed("Address", "address") \
    .withColumnRenamed("City", "city") \
    .withColumnRenamed("State", "state") \
    .withColumnRenamed("Zip", "zip") \
    .withColumnRenamed("Inspection Date", "inspection_date") \
    .withColumnRenamed("Inspection Type", "inspection_type") \
    .withColumnRenamed("Results", "results") \
    .withColumnRenamed("Violations", "violations") \
    .withColumnRenamed("Latitude", "latitude") \
    .withColumnRenamed("Longitude", "longitude") \
    .withColumnRenamed("Location", "location")

# Write to Delta table — this is where NOT NULL constraints are actually enforced
TABLE_NAME = "students_data.`team4-chicago`.food_inspections"

df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

# Read back from Delta to confirm nullable flags are enforced
df_delta = spark.table(TABLE_NAME)
print(f"Rows: {df_delta.count():,}")
df_delta.printSchema()
display(df_delta.limit(20))

In [0]:
TABLE_NAME = "students_data.`team4-chicago`.food_inspections"

# Columns that must not be null
not_null_columns = [
    "inspection_id", "dba_name", "license_num", "facility_type",
    "risk", "address", "city", "state", "zip",
    "inspection_date", "inspection_type", "results"
]

for col_name in not_null_columns:
    spark.sql(f"ALTER TABLE {TABLE_NAME} ALTER COLUMN {col_name} SET NOT NULL")
    print(f"SET NOT NULL: {col_name}")

print("\nDone. Verifying schema:")
spark.table(TABLE_NAME).printSchema()